<a href="https://colab.research.google.com/github/artursafrastyan-uni/nlp_project/blob/main/nlp_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Set Up**

In [ ]:
!pip install -q transformer_lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.2 MB/s eta 0:00:00


In [ ]:
import torch
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens import HookedTransformer
device = "cuda" if torch.cuda.is_available() else "cpu"

Using device: cuda


/tmp/ipykernel_4492/3614160758.py:3: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  import transformer_lens.utils as utils


# **Loading GPT 2 small**

In [ ]:
# fold_ln=True and center_writing_weights=True as these are considered standard
# practices in mechanistic interpretability to make the model easier to analyze.
model = HookedTransformer.from_pretrained(
    "gpt2-small",
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    refactor_factored_attn_matrices=True,
    device=device
)
print("Model loaded successfully!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer
Model loaded successfully!


In [ ]:
print(f"Number of layers (n_layers): {model.cfg.n_layers}")
print(f"Number of heads per layer (n_heads): {model.cfg.n_heads}")
print(f"Hidden dimension size (d_model): {model.cfg.d_model}")
print(f"Vocabulary size (d_vocab): {model.cfg.d_vocab}")


Number of layers (n_layers): 12
Number of heads per layer (n_heads): 12
Hidden dimension size (d_model): 768
Vocabulary size (d_vocab): 50257


# **Dataset creation**

In [ ]:
import random
from transformers import GPT2Tokenizer

#Generates the A -> A synthetic dataset for induction head analysis.Creates a repeated sequence and a corrupted sequence where the final target token is resampled.
def generate_synthetic_dataset(tokenizer, num_samples=200, seq_len=10):
    vocab_size = tokenizer.vocab_size
    clean_sequences = []
    corrupted_sequences = []
    for _ in range(num_samples):
        # Generate random sequence A
        A = [random.randint(0, 50000) for _ in range(seq_len)]
        # Clean sequence: A followed by A
        clean_seq = A + A
        # Corrupted sequence: A followed by A, but the last token is randomly resampled
        corrupted_seq = A + A[:-1]
        # Resample the target token to break the pattern
        resampled_target = random.randint(0, 50000)
        while resampled_target == A[-1]:
            resampled_target = random.randint(0, 50000)

        corrupted_seq.append(resampled_target)
        clean_sequences.append(clean_seq)
        corrupted_sequences.append(corrupted_seq)

    return torch.tensor(clean_sequences), torch.tensor(corrupted_sequences)

#Generates the templated Indirect Object Identification dataset.
#Clean: [Name A] and [Name B] went to the [Place]. [Name B] gave a [Object] to Corrupted: [Name A] and [Name B] went to the [Place]. [Name C] gave a [Object] to
def generate_ioi_dataset(tokenizer, num_samples=200):
    # Using words that are guaranteed to be single tokens in GPT-2 when prefixed with a space
    names = [" John", " Mary", " Bob", " Alice", " Tom", " Sarah", " Paul", " Emma"]
    places = [" store", " park", " school", " house", " beach", " mall", " zoo", " bank"]
    objects = [" book", " ball", " ring", " pen", " dog", " cat", " car", " gift"]
    clean_prompts = []
    corrupted_prompts = []
    for _ in range(num_samples):
        name_a, name_b = random.sample(names, 2)
        place = random.choice(places)
        obj = random.choice(objects)

        # Clean prompt
        clean_text = f"{name_a} and{name_b} went to the{place}.{name_b} gave a{obj} to"
        # Corrupted prompt: Random name flip for the subject of the second sentence
        available_corrupted_names = [n for n in names if n not in [name_a, name_b]]
        name_c = random.choice(available_corrupted_names)
        corrupted_text = f"{name_a} and{name_b} went to the{place}.{name_c} gave a{obj} to"
        clean_prompts.append(clean_text)
        corrupted_prompts.append(corrupted_text)

    tokenizer.pad_token = tokenizer.eos_token
    clean_tokens = tokenizer(clean_prompts, return_tensors="pt", padding=True).input_ids
    corrupted_tokens = tokenizer(corrupted_prompts, return_tensors="pt", padding=True).input_ids
    return clean_tokens, corrupted_tokens

# **Analysis of Attention Patterns**

In [ ]:
import os
# Initializing tokenizer if not already done
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

dataset_path = "/content/synthetic_dataset.pt"

# Loading the pre-computed synthetic dataset or generate if not found
if not os.path.exists(dataset_path):
    print(f"Dataset not found at {dataset_path}. Generating now...")
    clean_sequences_gen, _ = generate_synthetic_dataset(tokenizer, num_samples=200, seq_len=10)
    # Save the clean sequences to the expected path
    torch.save({"clean": clean_sequences_gen}, dataset_path)
    print(f"Dataset generated and saved to {dataset_path}.")

data = torch.load(dataset_path, map_location=device)
clean_sequences = data["clean"]
# The sequence is formatted as A -> A. We can find the length of A by halving the total length.
total_seq_len = clean_sequences.shape[1]
N = total_seq_len // 2
# I ran the forward pass without calculating gradients to save memory
with torch.no_grad():
    logits, cache = model.run_with_cache(clean_sequences)

num_layers = model.cfg.n_layers
num_heads = model.cfg.n_heads
induction_scores = torch.zeros((num_layers, num_heads))
for layer in range(num_layers):
    attn_pattern = cache[f"blocks.{layer}.attn.hook_pattern"]
    score_for_layer = torch.zeros(num_heads, device=device)
    valid_positions = 0

    # I analyze the second half of the sequence where the pattern A is repeating.
    for i in range(N, total_seq_len):
        src_target = i - N + 1
        if src_target < total_seq_len:
            attention_weights = attn_pattern[:, :, i, src_target]
            score_for_layer += attention_weights.mean(dim=0)
            valid_positions += 1

    induction_scores[layer] = (score_for_layer / valid_positions).cpu()

print("Top Candidate Induction Heads:")
top_scores, top_indices = torch.topk(induction_scores.flatten(), 10)

for rank, (score, idx) in enumerate(zip(top_scores, top_indices)):
    layer = idx.item() // num_heads
    head = idx.item() % num_heads
    print(f"{rank + 1}. Layer {layer}, Head {head} | Average Induction Attention: {score.item():.4f}")

Dataset not found at /content/synthetic_dataset.pt. Generating now...
Dataset generated and saved to /content/synthetic_dataset.pt.

--- Top Candidate Induction Heads ---
1. Layer 5, Head 5 | Average Induction Attention: 0.9150
2. Layer 5, Head 1 | Average Induction Attention: 0.8414
3. Layer 7, Head 10 | Average Induction Attention: 0.8270
4. Layer 6, Head 9 | Average Induction Attention: 0.8223
5. Layer 7, Head 2 | Average Induction Attention: 0.6335
6. Layer 10, Head 7 | Average Induction Attention: 0.5931
7. Layer 5, Head 0 | Average Induction Attention: 0.5366
8. Layer 10, Head 1 | Average Induction Attention: 0.5321
9. Layer 9, Head 6 | Average Induction Attention: 0.4852
10. Layer 11, Head 10 | Average Induction Attention: 0.4435


# **Direct Logit Attribution**

In [ ]:
# I have tested the top head I found: Layer 5, Head 5
target_layer = 5
target_head = 5
# the sequence is fed up to the second-to-last token to predict the final token
inputs = clean_sequences[:, :-1]
targets = clean_sequences[:, -1]
# no_grad() was used to save memory
with torch.no_grad():
    # Cache 'hook_z' which contains the head activations BEFORE the W_O projection
    logits, clean_cache = model.run_with_cache(inputs, names_filter=lambda name: name.endswith("attn.hook_z"))
z = clean_cache[f"blocks.{target_layer}.attn.hook_z"][:, -1, target_head, :]

# Getting the specific W_O (Output) matrix for Layer 5, Head 5
W_O_head = model.blocks[target_layer].attn.W_O[target_head]

# Manually projected z through W_O to get the final head_output
head_output = z @ W_O_head
# The Logit Lens applied. Projected this specific head's output directly into vocabulary space
direct_logits = head_output @ model.W_U
# Measure how strongly L5H5 is pushing for the correct target token vs all other tokens
target_logits = direct_logits.gather(-1, targets.unsqueeze(-1)).squeeze()
mean_logits = direct_logits.mean(dim=-1)

print(f"Average logit for the correct target token: {target_logits.mean().item():.4f}")
print(f"Average logit for all other tokens:         {mean_logits.mean().item():.4f}")

Average logit for the correct target token: 4.7380
Average logit for all other tokens:         0.0000


In [ ]:
# To prove causality, I have dynamically corrupted the input by changing the "source" token at index N-1.
# This ensures the base model naturally predicts the wrong (corrupted) token.
corrupted_inputs = inputs.clone()
corrupted_targets = torch.randint(0, model.cfg.d_vocab, (inputs.shape[0],), device=inputs.device)
# Ensure the new target is actually different from the original
corrupted_targets = torch.where(corrupted_targets == targets, corrupted_targets + 1, corrupted_targets)
corrupted_inputs[:, N-1] = corrupted_targets

# Baseline Corrupted Run (No patching)
with torch.no_grad():
    corrupted_logits = model(corrupted_inputs)
corr_prob = corrupted_logits[:, -1, :].softmax(dim=-1).gather(-1, targets.unsqueeze(-1)).mean()
def patch_l5h5_hook(corrupted_activation, hook):
    # 'hook_z' contains the head activations BEFORE the W_O projection
    # We surgically overwrite ONLY Head 5's activation with the clean cached activation
    corrupted_activation[:, :, target_head, :] = clean_cache[hook.name][:, :, target_head, :]
    return corrupted_activation
# I ran the model on the corrupted input, but trigger the hook at Layer 5
with torch.no_grad():
    patched_logits = model.run_with_hooks(
        corrupted_inputs,
        fwd_hooks=[(f"blocks.{target_layer}.attn.hook_z", patch_l5h5_hook)]
    )
patched_prob = patched_logits[:, -1, :].softmax(dim=-1).gather(-1, targets.unsqueeze(-1)).mean()

# Clean Baseline Run (For comparison)
clean_prob = logits[:, -1, :].softmax(dim=-1).gather(-1, targets.unsqueeze(-1)).mean()

print(f"Probability of the model predicting the ORIGINAL target token:")
print(f"1. Clean Input (Baseline):    {clean_prob.item():.4f}")
print(f"2. Corrupted Input:           {corr_prob.item():.4f}")
print(f"3. Corrupted + Patched L5H5:  {patched_prob.item():.4f}")


Probability of the model predicting the ORIGINAL target token:
1. Clean Input (Baseline):    0.5592
2. Corrupted Input:           0.0000
3. Corrupted + Patched L5H5:  0.0000


In [ ]:
# Extract the raw logit scores for the original target token
clean_target_logit = logits[:, -1, :].gather(-1, targets.unsqueeze(-1)).mean()
corr_target_logit = corrupted_logits[:, -1, :].gather(-1, targets.unsqueeze(-1)).mean()
patched_target_logit = patched_logits[:, -1, :].gather(-1, targets.unsqueeze(-1)).mean()

print(f"Logit Recovery of Original Target:")
print(f"1. Clean Logit (Baseline):   {clean_target_logit.item():.4f}")
print(f"2. Corrupted Logit:          {corr_target_logit.item():.4f}")
print(f"3. Corrupted + Patched L5H5: {patched_target_logit.item():.4f}")
recovery = patched_target_logit - corr_target_logit
print(f"\nCausal impact of patching L5H5: +{recovery.item():.4f} logits")


--- Logit Recovery of Original Target ---
1. Clean Logit (Baseline):   14.9469
2. Corrupted Logit:          0.0208
3. Corrupted + Patched L5H5: 0.4151

Causal impact of patching L5H5: +0.3942 logits


In [ ]:
induction_ensemble = [(5, 5), (5, 1), (6, 9), (7, 10)]
def patch_ensemble_hook(corrupted_activation, hook):
    layer = int(hook.name.split(".")[1])
    heads_in_layer = [h for l, h in induction_ensemble if l == layer]
    for head in heads_in_layer:
        corrupted_activation[:, :, head, :] = clean_cache[hook.name][:, :, head, :]

    return corrupted_activation
hooks_to_apply = [
    (f"blocks.{layer}.attn.hook_z", patch_ensemble_hook) for layer in [5, 6, 7]
]
with torch.no_grad():
    patched_logits_ensemble = model.run_with_hooks(
        corrupted_inputs,
        fwd_hooks=hooks_to_apply
    )

ensemble_target_logit = patched_logits_ensemble[:, -1, :].gather(-1, targets.unsqueeze(-1)).mean()
ensemble_recovery = ensemble_target_logit - corr_target_logit

print(f"Ensemble Logit Recovery:")
print(f"1. Clean Logit:              {clean_target_logit.item():.4f}")
print(f"2. Corrupted Logit:          {corr_target_logit.item():.4f}")
print(f"3. Ensemble Patched Logit:   {ensemble_target_logit.item():.4f}")
print(f"\nCausal impact of patching the Ensemble: +{ensemble_recovery.item():.4f} logits")


--- Ensemble Logit Recovery ---
1. Clean Logit:              14.9469
2. Corrupted Logit:          0.0208
3. Ensemble Patched Logit:   5.6921

Causal impact of patching the Ensemble: +5.6712 logits


# **Ablation Experiments on Abstract Pattern Recognition**

Following the methodology of the paper "Induction Heads as an Essential Mechanism for Pattern Matching in
In-context Learning", this experiment tests the causal impact of induction heads on Few-Shot In-Context Learning.

model's 5-shot ICL performance was evaluated on the **Repetition Task**. To prove causal dependence, targeted zero-ablations was performed on the induction ensemble identified in the above cells (L5H5, L5H1, L6H9, L7H10) and the resulting performance degradation was compared against a control ablation of randomly selected heads.


In [ ]:
import torch
import random
from collections import defaultdict
def generate_exact_induction_batch(model, batch_size=200, shots=5):
    words = [" cat", " dog", " apple", " tree", " house", " car", " bird", " book",
             " computer", " phone", " table", " chair", " sun", " moon", " water",
             " red", " blue", " green", " yellow", " black", " white", " fast", " slow"]

    prompts = []
    targets = []

    for _ in range(batch_size):
        prompt = ""
        for _ in range(shots):
            seq = random.sample(words, 4)
            seq_str = "".join(seq)
            prompt += f"{seq_str} ->{seq_str}\n"
        query_seq = random.sample(words, 4)
        query_str_full = "".join(query_seq)
        query_str_cut = "".join(query_seq[:3])
        prompt += f"{query_str_full} ->{query_str_cut}"
        prompts.append(prompt)
        targets.append(model.to_single_token(query_seq[3])) # Target is the 4th word

    return model.to_tokens(prompts), torch.tensor(targets, device=model.cfg.device)

def evaluate_icl_confidence(model, tokens, targets, fwd_hooks=[]):
    with torch.no_grad():
        logits = model.run_with_hooks(tokens, fwd_hooks=fwd_hooks)
    final_logits = logits[:, -1, :]
    target_probs = final_logits.softmax(dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze()
    return target_probs.mean().item() * 100
def get_ablation_hooks(ensemble_heads):
    # Creates a list of hooks that will set the activations of the specified heads to 0.
    hooks = []
    layer_to_heads = defaultdict(list)
    for layer, head in ensemble_heads:
        layer_to_heads[layer].append(head)

    for layer, heads in layer_to_heads.items():
        def ablation_hook(activation, hook, heads_to_ablate=heads):
            # 'hook_z' contains the head activations BEFORE the W_O projection
            for h in heads_to_ablate:
                activation[:, :, h, :] = 0.0
            return activation

        hooks.append((f"blocks.{layer}.attn.hook_z", ablation_hook))
    return hooks
tokens, targets = generate_exact_induction_batch(model, batch_size=200, shots=3)
baseline_conf = evaluate_icl_confidence(model, tokens, targets)
induction_heads = [
    (5, 5), (5, 1), (7, 10), (6, 9), (7, 2),
    (10, 7), (5, 0), (10, 1), (9, 6), (11, 10)
]
ind_hooks = get_ablation_hooks(induction_heads)
ind_conf = evaluate_icl_confidence(model, tokens, targets, fwd_hooks=ind_hooks)
random_heads = [
    (6, 2), (7, 5), (8, 1), (9, 3), (10, 4),
    (11, 2), (6, 7), (7, 8), (8, 11), (9, 0)
]
rnd_hooks = get_ablation_hooks(random_heads)
rnd_conf = evaluate_icl_confidence(model, tokens, targets, fwd_hooks=rnd_hooks)
print(f"Full Model Baseline:    {baseline_conf:.1f}% Confidence")
print(f"Induction Head Ablation:  {ind_conf:.1f}% Confidence  ({ind_conf - baseline_conf:+.1f}%)")
print(f"Random Head Ablation:      {rnd_conf:.1f}% Confidence  ({rnd_conf - baseline_conf:+.1f}%)\n")

Full Model Baseline:       75.8% Confidence
Induction Head Ablation:   11.8% Confidence  (-64.0%)

Random Head Ablation:      68.1% Confidence  (-7.7%)



# **Indirect Object Identification (IOI) & Circuit Redundancy**

Following the methodology in *Wang et al. (Interpretability in the Wild)*, we now switch to the templated IOI dataset that was generated above (`templated_dataset.pt`).

The goal here is twofold:
1. **Identify Semantic Heads**: Direct Logit Attribution was used to verify that **Layer 9, Head 9** acts as a "Name Mover Head", responsible for copying the correct name (Name A) to the output.
2. **Prove Circuit Redundancy**: Section 3.4 of the paper was replicated. I zero-ablate the primary Name Mover Heads (9.9, 9.6, 10.0). We will observe that the model's confidence barely drops, proving that "Backup Name Mover Heads" dynamically take over the task.

In [ ]:
import os
ioi_dataset_path = "templated_dataset.pt"
if not os.path.exists(ioi_dataset_path):
    print(f"IOI dataset not found at {ioi_dataset_path}.")
    clean_ioi_tokens, corrupted_ioi_tokens = generate_ioi_dataset(model.tokenizer, num_samples=200)
    torch.save({"clean": clean_ioi_tokens, "corrupted": corrupted_ioi_tokens}, ioi_dataset_path)
ioi_data = torch.load(ioi_dataset_path, map_location=device)
clean_ioi = ioi_data["clean"]
corrupted_ioi = ioi_data["corrupted"]
ioi_targets = clean_ioi[:, 0]
ioi_inputs = clean_ioi
with torch.no_grad():
    ioi_logits, ioi_cache = model.run_with_cache(ioi_inputs)
target_layer = 9
target_head = 9 # Known Name Mover Head
z = ioi_cache[f"blocks.{target_layer}.attn.hook_z"][:, -1, target_head, :]
W_O_head = model.blocks[target_layer].attn.W_O[target_head]
head_output = z @ W_O_head
direct_logits = head_output @ model.W_U
target_logits = direct_logits.gather(-1, ioi_targets.unsqueeze(-1)).squeeze()
mean_logits = direct_logits.mean(dim=-1)
print(f"Average logit for correct Name A: {target_logits.mean().item():.4f}")
print(f"Average logit for all other tokens: {mean_logits.mean().item():.4f}")
#zero-ablation for the known primary Name Mover Heads
name_movers = [(9, 9), (9, 6), (10, 0)]

def ablate_name_movers_hook(activation, hook):
    layer = int(hook.name.split(".")[1])
    heads_in_layer = [h for l, h in name_movers if l == layer]
    for head in heads_in_layer:
        activation[:, :, head, :] = 0.0
    return activation
hooks_to_apply = [(f"blocks.{layer}.attn.hook_z", ablate_name_movers_hook) for layer in [9, 10]]
with torch.no_grad():
    with model.hooks(fwd_hooks=hooks_to_apply):
        ablated_logits, ablated_cache = model.run_with_cache(ioi_inputs)
clean_target_logit = ioi_logits[:, -1, :].gather(-1, ioi_targets.unsqueeze(-1)).mean()
ablated_target_logit = ablated_logits[:, -1, :].gather(-1, ioi_targets.unsqueeze(-1)).mean()
print(f"Clean Target Logit:               {clean_target_logit.item():.4f}")
print(f"Logit after Ablating Name Mvrs:   {ablated_target_logit.item():.4f}")

Average logit for correct Name A: 0.6687
Average logit for all other tokens: 0.0000
ircuit Redundancy
Clean Target Logit:               4.7316
Logit after Ablating Name Mvrs:   4.4870
